# Fine Tuning Comparison

**Experiment Note:** 

- **In this notebook, you have to restart kernel before training each model to clear memory, remove cache, and avoid OOM errors.**
- **Must Run Configurations Every times Before Fine Tuning The Model**

### Fine-Tuning Configurations

In [ ]:
onfig cell
# Fine-Tuning Notebook Configuration
import sys
import os
import json
sys.path.append("..")

# Fine-tuning hyperparameters
MAX_SEQ_LEN              = 384
PER_DEVICE_BATCH         = 8
GRAD_ACCUM               = 8
LEARNING_RATE            = 4e-4
NUM_EPOCHS_FT            = 20        # ceiling; early stopping normally stops at ~2-5
WARMUP_STEPS             = 163       # 2% of the 20-epoch steps for ~10M tokens
WEIGHT_DECAY             = 0.1
EARLY_STOPPING_PATIENCE  = 3
EARLY_STOPPING_THRESHOLD = 0.001

# Optimizer (AdamW)
ADAM_BETA1               = 0.9
ADAM_BETA2               = 0.95
ADAM_EPS                 = 1e-8
MAX_GRAD_NORM            = 1.0

# LR schedule
# DeepSeekMoE training approach: LR x 0.316 at 80% of steps, again at 90%.
SCHEDULE                 = "warmup_step_decay"   # "warmup_step_decay" | "cosine" | "linear" | "constant"
DECAY_FRACS              = (0.80, 0.90)
DECAY_GAMMA              = 0.316

# DeepSpeed Configuration for Fine-Tuning
ds_config = {
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": 5e8,
        "allgather_bucket_size": 5e8,
    },
    "gradient_clipping": 1.0,
    "steps_per_print": 10,
    "wall_clock_breakdown": False,
    "zero_allow_untested_optimizer": True,
}

ds_config_path = "./deepseekmoe_dynamic_routing_algorithms/ds_config.json"                 # Please change to your path to this repo
with open(ds_config_path, "w") as f:
    json.dump(ds_config, f, indent=2)

### DeepSeekMoE (Baseline)

In [ ]:
from source.fine_tuning_utils.fine_tuning_runner import run_fine_tuning
from source.fine_tuning_utils.config import PRETRAINED_BASELINE, OUTPUT_FT_BASELINE
from source.data_preprocessing.config import DEFAULT_FINE_PATH, DEFAULT_VAL_PATH

run_fine_tuning(
    PRETRAINED_BASELINE, OUTPUT_FT_BASELINE,
    
    # Dataset Paths
    data_file_path=DEFAULT_FINE_PATH,
    val_data_file_path=DEFAULT_VAL_PATH,

    # Fine-Tuning Hyperparameters
    MAX_SEQ_LEN=MAX_SEQ_LEN,
    PER_DEVICE_BATCH=PER_DEVICE_BATCH,
    GRAD_ACCUM=GRAD_ACCUM,
    LEARNING_RATE=LEARNING_RATE,
    NUM_EPOCHS_FT=NUM_EPOCHS_FT,
    WARMUP_STEPS=WARMUP_STEPS,
    WEIGHT_DECAY=WEIGHT_DECAY,
    EARLY_STOPPING_PATIENCE=EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_THRESHOLD=EARLY_STOPPING_THRESHOLD,

    # Optimizer (AdamW)
    ADAM_BETA1=ADAM_BETA1,
    ADAM_BETA2=ADAM_BETA2,
    ADAM_EPS=ADAM_EPS,
    MAX_GRAD_NORM=MAX_GRAD_NORM,

    # LR schedule
    SCHEDULE=SCHEDULE,
    DECAY_FRACS=DECAY_FRACS,
    DECAY_GAMMA=DECAY_GAMMA,

    # DeepSpeed Config
    ds_config='./deepseekmoe_dynamic_routing_algorithms/ds_config.json'                     # Please change to the same path as the saved ds_config.json file above
)

### DeepSeekMoE with DYNMoE Routing (Prototype)

In [ ]:
from source.fine_tuning_utils.fine_tuning_runner import run_fine_tuning
from source.fine_tuning_utils.config import PRETRAINED_DYNMOE_ROUTING, OUTPUT_FT_DYNMOE_ROUTING
from source.data_preprocessing.config import DEFAULT_FINE_PATH, DEFAULT_VAL_PATH

run_fine_tuning(
    PRETRAINED_DYNMOE_ROUTING, OUTPUT_FT_DYNMOE_ROUTING,
    
    # Dataset Paths
    data_file_path=DEFAULT_FINE_PATH,
    val_data_file_path=DEFAULT_VAL_PATH,

    # Fine-Tuning Hyperparameters
    MAX_SEQ_LEN=MAX_SEQ_LEN,
    PER_DEVICE_BATCH=PER_DEVICE_BATCH,
    GRAD_ACCUM=GRAD_ACCUM,
    LEARNING_RATE=LEARNING_RATE,
    NUM_EPOCHS_FT=NUM_EPOCHS_FT,
    WARMUP_STEPS=WARMUP_STEPS,
    WEIGHT_DECAY=WEIGHT_DECAY,
    EARLY_STOPPING_PATIENCE=EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_THRESHOLD=EARLY_STOPPING_THRESHOLD,

    # Optimizer (AdamW)
    ADAM_BETA1=ADAM_BETA1,
    ADAM_BETA2=ADAM_BETA2,
    ADAM_EPS=ADAM_EPS,
    MAX_GRAD_NORM=MAX_GRAD_NORM,

    # LR schedule
    SCHEDULE=SCHEDULE,
    DECAY_FRACS=DECAY_FRACS,
    DECAY_GAMMA=DECAY_GAMMA,

    # DeepSpeed Config
    ds_config='./deepseekmoe_dynamic_routing_algorithms/ds_config.json'                       # Please change to the same path as the saved ds_config.json file above
)

### Fine Tuning Comparison: DeepSeekMoE vs DeepSeekMoE with DYNMoE routing

In [ ]:
from source.fine_tuning.monitoring import (
    plot_training_comparison, print_monitoring_summary,
)
from source.fine_tuning.config import (
    OUTPUT_FT_BASELINE, OUTPUT_FT_DYNMOE_ROUTING,
    PER_DEVICE_BATCH, GRAD_ACCUM, MAX_SEQ_LEN, world_size,
)

TOKENS_PER_STEP_FT = PER_DEVICE_BATCH * GRAD_ACCUM * MAX_SEQ_LEN * max(world_size, 1)

res = plot_training_comparison(
    OUTPUT_FT_BASELINE,
    OUTPUT_FT_DYNMOE_ROUTING,
    output_path="finetuning_comparison.png",
    tokens_per_step=TOKENS_PER_STEP_FT,
    figsize=(15, 16),
    dpi=150,
    annotate_final=True,
    show=False,
)

print_monitoring_summary(res)